# `expansion_hunter_05_submit` — run the cohort

Open in a VWB JupyterLab app and **Run All**. Nothing to edit: the run comes from `ACTIVE_RUN`, written by
[`expansion_hunter_03_setup.ipynb`](expansion_hunter_03_setup.ipynb).

Each pass checks Batch, then submits at most one wave. What it submits, in order:

1. **Canary.** On a new code version (any change to the EH worker scripts), 200 samples, then nothing until
   they finish.
2. **Waves** of never-attempted samples, refilling to 10,000 in flight once fewer than 5,000 remain.
3. **Retries** of failed samples, once every sample has had a first attempt (up to 3 attempts each).

It **stops** (and says why) when:

- more than 5% of recent first attempts failed. Look at the logs in 06, push a fix (that starts a new canary),
  or raise `max_failure_rate` in 03 if the failures are expected;
- the next wave would push estimated spend past `budget_usd` (raise it in 03 to continue);
- every sample is done or out of attempts.

The last cell keeps going by itself every 20 minutes while this notebook stays open (`AUTOPILOT`). If the
kernel stops, nothing is lost: Run All again and it picks up from the ledger. Only one copy can submit at a
time (a lock in the run folder). Progress, cost and failure logs:
[`expansion_hunter_06_progress.ipynb`](expansion_hunter_06_progress.ipynb).

In [ ]:
from __future__ import annotations

import os
import shutil
import sys
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "scripts" / "vwb_batch" / "campaign.py").is_file():
            return p
        nested = p / "aou-lr-phase-2" / "scripts" / "vwb_batch" / "campaign.py"
        if nested.is_file():
            return nested.parents[2]
    raise FileNotFoundError("Cannot find scripts/vwb_batch/campaign.py. git pull kvg/aou-lr-phase-2 and re-run.")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "scripts"))
for name in [m for m in sys.modules if m == "vwb_batch" or m.startswith("vwb_batch.")]:
    del sys.modules[name]  # pick up a git pull without restarting the kernel

from vwb_batch import alloc, campaign, cohort, cost, gcs, registry  # noqa: E402
from vwb_batch.pipelines import expansion_hunter as eh  # noqa: E402

OUTPUT_BUCKET_GS = os.environ.get("EH_OUTPUT_BUCKET_GS", "gs://aou-lr-phase2-resources").rstrip("/")
PIPELINE = "expansion_hunter"
RUN_ID = os.environ.get("EH_RUN_ID", "").strip() or campaign.read_active_run(OUTPUT_BUCKET_GS, PIPELINE)
if not RUN_ID:
    raise SystemExit("No active run. Run expansion_hunter_03_setup first (or set EH_RUN_ID).")
PATHS = registry.run_paths(output_bucket=OUTPUT_BUCKET_GS, pipeline=PIPELINE, run_id=RUN_ID)
if shutil.which("gcloud") is None:
    raise SystemExit("gcloud is not on PATH. Open this notebook in a VWB Jupyter app.")
PROJECT, SA = alloc.require_vwb_env()
CODE_VERSION = eh.code_version()
print("RUN_ID:", RUN_ID, "   ledger:", PATHS.prefix)
print("code version:", CODE_VERSION, "   project:", PROJECT)

## One pass: check Batch, then submit the next wave if the rules allow

In [ ]:
def run_once():
    return campaign.tick(
        PATHS,
        project=PROJECT,
        code_version=eh.code_version(),
        pipeline_short="eh",
        make_builders=lambda cfg: eh.campaign_builders(cfg, project=PROJECT, sa=SA),
    )


last_plan, submitted, state, cfg = run_once()

## Autopilot

Repeats the pass every `INTERVAL_MIN` minutes until the run is finished or halted. Interrupt the kernel to
stop; Batch jobs already submitted keep running. Workbench may pause an idle app; a running cell counts as
activity, but check the app's auto-pause setting before leaving it overnight.

In [ ]:
import time

AUTOPILOT = True
INTERVAL_MIN = 20

while AUTOPILOT and last_plan.action not in ("halt", "finished"):
    print(f"\nnext pass at {time.strftime('%H:%M', time.localtime(time.time() + INTERVAL_MIN * 60))}")
    time.sleep(INTERVAL_MIN * 60)
    print(f"--- {registry.utcnow()}")
    try:
        last_plan, submitted, state, cfg = run_once()
    except campaign.LockHeld as exc:
        print("skipped:", exc)
    except Exception as exc:  # a gcloud hiccup should not end an overnight run
        print("pass failed, will retry next interval:", repr(exc)[:500])
print("autopilot stopped:", last_plan.explain())